In [2]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent
load_dotenv()


/home/rajthegreat/Documents/development/practise/agentic-ai/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
PRODUCTS = {
    "wireless headphones": {"price": 79.99,  "rating": 4.6, "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."},
	"smart watch":         {"price": 199.99, "rating": 4.3, "description": "Tracks heart rate and sleep. 5-day battery, water-resistant."},
	"mechanical keyboard": {"price": 129.00, "rating": 4.8, "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."},
	"laptop stand":        {"price": 34.99,  "rating": 4.5, "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat."},
    }

@tool
def get_product(name:str)-> str:
    """Look up a product by name and return its price, rating, stock, and desciption."""
    p=PRODUCTS.get(name.lower())
    if not p:
        return f"Product not found. Available:{', '.join(PRODUCTS)}"
    return str(p)

llm =ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

agent=create_agent(
    llm,
    tools=[get_product],
    system_prompt="You are a helpful product assistant for an online tech store.",
)


In [4]:
def ask(question: str):
    result=agent.invoke({"messages":[{"role":"user","content":question}]})
    print(result["messages"][-1].content)

In [5]:
ask("what is the price of wireless headphones.")

The price of the wireless headphones is $79.99.


In [6]:
REVIEWS = {
"wireless headphones": {"reviews": 1262, "rating": 4.6},
   "smart watch":         {"reviews": 340,  "rating": 3.9},
    "mechanical keyboard": {"reviews": 67,   "rating": 4.8},
    "laptop stand":        {"reviews": 781,  "rating": 4.5},
}

@tool
def get_review(name:str)-> str:
    """Look up a product review by a product name. Return the product name, number of review and rating"""
    r=REVIEWS.get(name.lower())
    if not r:
        return f"Review not available for this product"
    return str(r)

In [8]:
ask2("how do people like smart watch")

People generally like smartwatches, with an average rating of 4.3 out of 5 stars and over 340 reviews. The smartwatch is priced at $199.99 and has features such as tracking heart rate and sleep, a 5-day battery life, and is water-resistant.


In [9]:
ask2("what is the price and review of smart watch")

The price of the smart watch is $199.99. It has a rating of 4.3 and the description is "Tracks heart rate and sleep. 5-day battery, water-resistant." 

The product has 340 reviews with an average rating of 3.9.


In [11]:
ask("what is the price and review of smart watch")

The price of the smart watch is $199.99 and it has a rating of 4.3 out of 5. It tracks heart rate and sleep, has a 5-day battery life, and is water-resistant.


In [12]:
ask("what are the reviews on this product")

The wireless headphones have a rating of 4.6 out of 5 stars. 

Would you like to know the reviews for any other product?


In [20]:
from langgraph.checkpoint.memory import InMemorySaver
llm=ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
agent2=create_agent(
    llm,
    tools=[get_product, get_review],
    system_prompt="You are a helpful product assistant for an online tech store.",
    checkpointer=InMemorySaver()
)

def ask2(question: str):
    config={"configurable": {"thread_id":"user-alice-session-1"}}
    result=agent2.invoke(
        {"messages":[{"role":"user","content":question}]},
        config
    )
    print(result["messages"][-1].content)

In [21]:
ask2("what is the price of the wireless headphones.")

The price of the wireless headphones is $79.99.


In [22]:
ask2("what is its review?")

The wireless headphones have 1262 reviews with an average rating of 4.6.


In [18]:
ask2("what are the reviews on this product")

I need to know the name of the product you're looking for. Could you please provide the product name? I'll be happy to look up the reviews for you.
